# 01 – Exploratory Data Analysis
**Digital Ad Campaign Measurement & Attribution Framework**

This notebook explores the 200K+ ad event dataset covering impressions, clicks, and conversions across three targeting strategies:
- **Addressable** – user-level / cookie-based targeting
- **Cohort-based** – aggregated audience segments
- **Contextual** – page-content signals, no user identity


In [ ]:
import os, sys
sys.path.insert(0, os.path.join('..', ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Generate data if not present
data_path = '../data/ad_events.parquet'
if not os.path.exists(data_path):
    from data.generate_data import generate_ad_events, save_data
    df = generate_ad_events()
    save_data(df)
else:
    df = pd.read_parquet(data_path)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date']      = df['timestamp'].dt.date
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Basic Statistics ===')
print(f'Total events      : {len(df):>10,}')
print(f'Unique users      : {df.user_id.nunique():>10,}')
print(f'Total clicks      : {df.clicked.sum():>10,}  CTR={df.clicked.mean()*100:.2f}%')
print(f'Total conversions : {df.converted.sum():>10,}  CVR={df.converted.mean()*100:.2f}%')
print(f'Total revenue     : ${df.order_value_usd.sum():>12,.2f}')
print(f'Total spend       : ${df.ad_spend_usd.sum():>12,.2f}')
print(f'Overall ROAS      : {df.order_value_usd.sum()/df.ad_spend_usd.sum():.2f}x')
print(f'Date range        : {df.timestamp.min().date()} → {df.timestamp.max().date()}')

## 2. KPIs by Targeting Strategy

In [ ]:
strat = df.groupby('targeting_strategy').agg(
    impressions=('event_id',       'count'),
    clicks     =('clicked',        'sum'),
    conversions=('converted',      'sum'),
    revenue    =('order_value_usd','sum'),
    spend      =('ad_spend_usd',   'sum'),
).assign(
    CTR  = lambda d: d.clicks      / d.impressions,
    CVR  = lambda d: d.conversions / d.clicks.clip(1),
    ROAS = lambda d: d.revenue     / d.spend.clip(1e-6),
).round(4)

strat

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['CTR', 'CVR', 'ROAS']):
    strat[metric].plot.bar(ax=ax, color=['#2563EB','#10B981','#F59E0B'])
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('KPIs by Targeting Strategy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Temporal Trends

In [ ]:
daily = df.groupby('date').agg(
    revenue=('order_value_usd','sum'),
    spend  =('ad_spend_usd',  'sum'),
    clicks =('clicked',       'sum'),
    convs  =('converted',     'sum'),
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax1.plot(daily.date, daily.revenue, label='Revenue', color='#2563EB')
ax1.plot(daily.date, daily.spend,   label='Spend',   color='#F59E0B', linestyle='--')
ax1.axvline(pd.Timestamp('2024-02-15'), color='red', linestyle=':', label='A/B Start')
ax1.set_ylabel('USD'); ax1.legend(); ax1.set_title('Daily Revenue vs Spend')

ax2.plot(daily.date, daily.convs, label='Conversions', color='#10B981')
ax2.axvline(pd.Timestamp('2024-02-15'), color='red', linestyle=':')
ax2.set_ylabel('Count'); ax2.legend(); ax2.set_title('Daily Conversions')

plt.tight_layout()
plt.show()

## 4. Audience Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
features = ['recency_days','frequency','prior_clicks','context_score','bid_price_cpm','cohort_size']
for ax, feat in zip(axes.flat, features):
    df[feat].hist(bins=40, ax=ax, color='#2563EB', alpha=0.7, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('')
plt.suptitle('Audience Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
num_cols = ['recency_days','frequency','prior_clicks','context_score',
            'bid_price_cpm','cohort_size','clicked','converted']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. ROAS Heatmap: Ad Format × Device

In [ ]:
roas_pivot = (
    df.groupby(['ad_format','device_type'])
    .apply(lambda g: g.order_value_usd.sum() / g.ad_spend_usd.sum(), include_groups=False)
    .reset_index(name='ROAS')
    .pivot(index='ad_format', columns='device_type', values='ROAS')
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(roas_pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax,
            linewidths=0.5, annot_kws={'size': 10})
ax.set_title('ROAS by Ad Format × Device', fontweight='bold')
plt.tight_layout()
plt.show()